In [1]:
import os
from tools import *

import ants
import SimpleITK as sitk

print(f'AntsPy version = {ants.__version__}')
print(f'SimpleITK version = {sitk.__version__}')

AntsPy version = 0.6.3
SimpleITK version = 2.4.0


In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
print(f'project folder = {BASE_DIR}')

project folder = /home/folkk2/img_group_project/test-MRI-preprocessing-technique


In [18]:
raw_img_path = os.path.join(BASE_DIR, 'assets', 'raw_examples', 'fsl-open-dev_sub-001_T1w.nii.gz')
print(f'raw_img_path = {raw_img_path}')
raw_example = 'fsl-open-dev_sub-001_T1w.nii.gz'

raw_img_path = /home/folkk2/img_group_project/test-MRI-preprocessing-technique/assets/raw_examples/fsl-open-dev_sub-001_T1w.nii.gz


### AntsPy

In [4]:
raw_img_ants = ants.image_read(raw_img_path)

In [5]:
print(raw_img_ants)

ANTsImage (LPI)
	 Pixel Type : float (float32)
	 Components : 1
	 Dimensions : (160, 192, 192)
	 Spacing    : (1.2, 1.25, 1.25)
	 Origin     : (98.1114, 89.5975, -129.5975)
	 Direction  : [-1.  0.  0.  0. -1.  0.  0.  0.  1.]



### SimpleITK

In [6]:
raw_img_sitk = sitk.ReadImage(raw_img_path, sitk.sitkFloat32)

In [7]:
print(raw_img_sitk)

Image (0xc101680)
  RTTI typeinfo:   itk::Image<float, 3u>
  Reference Count: 1
  Modified Time: 1838
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 1812
  UpdateMTime: 1834
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [160, 192, 192]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [160, 192, 192]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [160, 192, 192]
  Spacing: [1.2, 1.25, 1.25]
  Origin: [98.1114, 89.5975, -129.597]
  Direction: 
-1 0 0
0 -1 0
0 0 1

  IndexToPointMatrix: 
-1.2 0 0
0 -1.25 0
0 0 1.25

  PointToIndexMatrix: 
-0.833333 0 0
0 -0.8 0
0 0 0.8

  Inverse Direction: 
-1 0 0
0 -1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (0xad22380)
      RTTI typeinfo:   itk::ImportImageContainer<unsigned long, float>
      Reference Count: 

In [8]:
def resample_to_isotropic(img_path):
    # 1. Read the image
    img = sitk.ReadImage(img_path, sitk.sitkFloat32)
    
    # 2. Get original spacing and size
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    
    print(f"Original Spacing: {original_spacing}")
    print(f"Original Size: {original_size}")
    
    # 3. Define the target spacing (1x1x1 mm)
    target_spacing = [1.2, 1.2, 1.2]
    
    # 4. Calculate the new size to maintain the same physical bounding box
    # New Size = Original Size * (Original Spacing / Target Spacing)
    new_size = [
        int(round(osz * ospc / tspc)) 
        for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)
    ]
    
    print(f"Target Spacing: {target_spacing}")
    print(f"New Size: {new_size}")
    
    # 5. Set up the resampler
    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(target_spacing)
    
    # Crucial: Keep the original physical orientation and origin!
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(img.GetPixelIDValue())
    
    # Use BSpline or Linear interpolation for standard MRIs.
    # (Use sitk.sitkNearestNeighbor if resampling a segmentation mask!)
    resampler.SetInterpolator(sitk.sitkBSpline) 
    
    # 6. Execute resampling
    resampled_img = resampler.Execute(img)
    
    # Convert back to numpy array for visualization with ipywidgets
    # Note: SimpleITK returns arrays in (Z, Y, X) order, so we transpose to (X, Y, Z)
    resampled_array = sitk.GetArrayFromImage(resampled_img)
    
    return resampled_img, resampled_array

In [9]:
reimg, reimg_arr = resample_to_isotropic(raw_img_path)

Original Spacing: (1.2000000476837158, 1.25, 1.25)
Original Size: (160, 192, 192)
Target Spacing: [1.2, 1.2, 1.2]
New Size: [160, 200, 200]


In [10]:
explore_3D_array(reimg_arr)

interactive(children=(IntSlider(value=99, description='SLICE', max=199), Output()), _dom_classes=('widget-inte…

In [11]:
resampling_img = sitk.DICOMOrient(reimg,'LPS')
print(resampling_img.GetDirection())
resampling_img_arr = sitk.GetArrayFromImage(resampling_img)
print(resampling_img_arr.shape)
explore_3D_array(resampling_img_arr)

(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
(200, 200, 160)


interactive(children=(IntSlider(value=99, description='SLICE', max=199), Output()), _dom_classes=('widget-inte…

In [12]:
resampling_ants = ants.from_sitk(resampling_img)

transformed = ants.denoise_image(resampling_ants, shrink_factor=1)

explore_3D_array_comparison(
    arr_before=sitk.GetArrayFromImage(ants.to_sitk(resampling_ants)),
    arr_after=sitk.GetArrayFromImage(ants.to_sitk(transformed)),
    cmap='grey'
)

interactive(children=(IntSlider(value=99, description='SLICE', max=199), Output()), _dom_classes=('widget-inte…

In [13]:
sitk_img_denoised = ants.to_sitk(transformed)
sitk_img_denoised_arr = sitk.GetArrayFromImage(ants.to_sitk(transformed))

In [20]:
out_folder =  os.path.join(BASE_DIR, 'assets', 'test-preprocessed')
out_folder = os.path.join(out_folder, raw_example.split('.')[0]) # create folder with name of the raw file
os.makedirs(out_folder, exist_ok=True) # create folder if not exists

out_filename = add_suffix_to_filename(raw_example, suffix='denoised')
out_path = os.path.join(out_folder, out_filename)

print(raw_img_path[len(BASE_DIR):])
print(out_path[len(BASE_DIR):])

/assets/raw_examples/fsl-open-dev_sub-001_T1w.nii.gz
/assets/test-preprocessed/fsl-open-dev_sub-001_T1w/fsl-open-dev_sub-001_T1w_denoised.nii.gz


In [21]:
sitk.WriteImage(sitk_img_denoised, out_path)